# Retrieval Failure Pattern Catalog

Treat this notebook like a workbench for common misses.

It is not here to prove that Kayak wins everywhere.
It is here to help the learner notice that different retrieval failures need different fixes.

The three patterns here are:

1. compressed matching hides the right evidence
2. the evidence is split across retrieval units
3. the candidate stage drops the oracle before reranking begins

Backstage verification note: the local deterministic rank-order claims in this notebook are covered by `python.tests.test_course_failure_patterns_smoke` and `python.tests.test_course_rag_debugging_smoke`.

In [ ]:
from pathlib import Path
import sys

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "python" / "kayak").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "python"))

import kayak

print("Working from repo root:", REPO_ROOT)
print("Backends available here:", kayak.available_backends())

In [ ]:
DIM = 64
TOKEN_TO_INDEX: dict[str, int] = {}


def token_vector(token: str) -> np.ndarray:
    index = TOKEN_TO_INDEX.setdefault(token, len(TOKEN_TO_INDEX))
    if index >= DIM:
        raise ValueError("Increase DIM for this notebook example.")
    vector = np.zeros(DIM, dtype=np.float32)
    vector[index] = np.float32(1.0)
    return vector


def encode_tokens(tokens: list[str]) -> np.ndarray:
    return np.stack([token_vector(token) for token in tokens])


def dense_mean(tokens: list[str]) -> np.ndarray:
    return encode_tokens(tokens).mean(axis=0)


def cosine_similarity(left: np.ndarray, right: np.ndarray) -> float:
    return float(np.dot(left, right) / (np.linalg.norm(left) * np.linalg.norm(right)))


## Pattern 1: Compressed Matching Hides The Evidence

This is the familiar feeling of, "the answer is in there, why is this distractor winning?"
The relevant document contains both evidence tokens but also a lot of noise.
A partial-overlap distractor repeats only one token loudly enough to win under compression.


In [ ]:
query_tokens = ["cancel", "subscription"]
documents = {
    "doc-relevant": query_tokens + [f"noise-{i}" for i in range(20)],
    "doc-partial": ["cancel", "cancel", "cancel", "cancel"],
    "doc-other": ["billing", "invoice"],
}

index = kayak.documents(
    list(documents.keys()),
    [encode_tokens(tokens) for tokens in documents.values()],
).pack()
query = kayak.query(encode_tokens(query_tokens))

kayak_hits = kayak.search(query, index, k=3, backend=kayak.NUMPY_REFERENCE_BACKEND)
dense_scores = sorted(
    (
        (doc_id, cosine_similarity(dense_mean(query_tokens), dense_mean(tokens)))
        for doc_id, tokens in documents.items()
    ),
    key=lambda row: row[1],
    reverse=True,
)

print("If I keep full token interaction:", [(hit.doc_id, round(hit.score, 3)) for hit in kayak_hits])
print("If I use the compressed shortcut:", [(doc_id, round(score, 3)) for doc_id, score in dense_scores])

## Pattern 2: The Evidence Is Split Across Retrieval Units

This one feels different.
Exact late interaction does not save you if no single retrieval unit actually contains the full evidence.

The practical lesson is simple:

- the scorer may be fine
- the retrieval unit may be wrong


In [ ]:
query = kayak.query(encode_tokens(["cancel", "subscription"]))

split_index = kayak.documents(
    ["chunk-a", "chunk-b", "other"],
    [
        encode_tokens(["cancel"]),
        encode_tokens(["subscription"]),
        encode_tokens(["invoice"]),
    ],
).pack()

grouped_index = kayak.documents(
    ["doc-combined", "other"],
    [
        encode_tokens(["cancel", "subscription"]),
        encode_tokens(["invoice"]),
    ],
).pack()

split_hits = kayak.search(query, split_index, k=3, backend=kayak.NUMPY_REFERENCE_BACKEND)
grouped_hits = kayak.search(query, grouped_index, k=2, backend=kayak.NUMPY_REFERENCE_BACKEND)

print("If I keep the evidence split apart:", [(hit.doc_id, hit.score) for hit in split_hits])
print("If I regroup the evidence into one unit:", [(hit.doc_id, hit.score) for hit in grouped_hits])

## Pattern 3: The Candidate Stage Drops The Oracle

This is the shortlist problem in its cleanest form.
If the oracle document never enters the candidate set, exact reranking cannot recover it.

Here we compare an exact full scan with two document-proxy plans:

- one narrow plan that loses the oracle
- one slightly wider plan that lets exact stage 2 recover the right hit


In [ ]:
query = kayak.query(encode_tokens(["cancel", "subscription"]))
index = kayak.documents(
    ["doc-partial", "doc-relevant", "doc-other"],
    [
        encode_tokens(["cancel", "cancel", "cancel"]),
        encode_tokens(["cancel", "subscription"]),
        encode_tokens(["invoice"]),
    ],
).pack()

exact_result = kayak.search_with_plan(
    query,
    index,
    kayak.exact_full_scan_search_plan(final_k=1, candidate_k=3),
    backend=kayak.NUMPY_REFERENCE_BACKEND,
)

narrow_result = kayak.search_with_plan(
    query,
    index,
    kayak.document_proxy_search_plan(
        final_k=1,
        candidate_k=1,
        query_vector_budget=1,
        document_vector_budget=1,
    ),
    backend=kayak.NUMPY_REFERENCE_BACKEND,
)

wider_result = kayak.search_with_plan(
    query,
    index,
    kayak.document_proxy_search_plan(
        final_k=1,
        candidate_k=2,
        query_vector_budget=1,
        document_vector_budget=1,
    ),
    backend=kayak.NUMPY_REFERENCE_BACKEND,
)

print("If I keep exact search, the top hit is:", [hit.doc_id for hit in exact_result.hits])
print("With a too-narrow stage 1, the candidates are:", narrow_result.candidate_stage.candidate_doc_ids)
print("So the final hit becomes:", [hit.doc_id for hit in narrow_result.hits])
print("If I widen stage 1 a little, the candidates become:", wider_result.candidate_stage.candidate_doc_ids)
print("And the final hit recovers to:", [hit.doc_id for hit in wider_result.hits])

## Practical Takeaway

These three misses may all look like "retrieval is bad," but they want different next moves:

- if compressed matching hides the evidence, debug against exact late interaction
- if the evidence is split across retrieval units, change the retrieval unit
- if the candidate stage drops the oracle, widen or improve stage 1

That is why the course should teach diagnosis before optimization.